# Encrypted Statistics Tutorial

This notebook covers the `stats.py` module, demonstrating how to compute statistical metrics over encrypted arrays. Since division and dynamic branching are complex in FHE, these functions utilize specialized mathematical approximations and tournament reductions to achieve exact or bounded results.

## Basic Statistical Metrics
Functions covered: `array_mean`, `array_variance`, `array_std`, `array_covariance`

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.stats import array_mean, array_variance, array_std, array_covariance

def test_basic_stats(a: list[int], b: list[int]):
    mean = array_mean(a)
    variance = array_variance(a)
    # std requires bounds because it computes a square root via lookup tables
    std = array_std(a, min_value=0, max_value=50)
    cov = array_covariance(a, b)
    return mean, variance, std, cov

# 1. Cleartext Execution
a = [10, 20, 30, 40]
b = [1, 2, 3, 4]
res_mean, res_var, res_std, res_cov = test_basic_stats(a, b)

assert res_mean == 25  # (10+20+30+40)//4 = 100//4 = 25
assert res_var == 125  # variance of [10, 20, 30, 40]
assert res_std == 11   # int(sqrt(125)) = 11
assert res_cov == 12   # covariance
print("Cleartext basic stats passed!")

# 2. FHE Compilation
compiler = fhe.Compiler(test_basic_stats, {"a": "encrypted", "b": "encrypted"})
inputset = [([0, 0, 0, 0], [0, 0, 0, 0]), ([10, 20, 30, 40], [1, 2, 3, 4])]
circuit = compiler.compile(inputset)

# 3. Encrypted Execution & Verification
enc_res = circuit.encrypt_run_decrypt(a, b)
assert enc_res[0] == res_mean
assert enc_res[1] == res_var
assert enc_res[2] == res_std
assert enc_res[3] == res_cov
print("✅ Encrypted basic stats passed!")

## Extremes and Spread
Functions covered: `array_max`, `array_min`, `array_range`, `array_count_greater`

In [ ]:
from concrete_fhe_toolkit.stats import array_max, array_min, array_range, array_count_greater

def test_extremes(a: list[int], threshold: int):
    max_val = array_max(a)
    min_val = array_min(a)
    rng = array_range(a)
    count_gt = array_count_greater(a, threshold)
    return max_val, min_val, rng, count_gt

a = [15, 5, 25, 10]
threshold = 12

res_max, res_min, res_rng, res_count = test_extremes(a, threshold)

assert res_max == 25
assert res_min == 5
assert res_rng == 20  # 25 - 5
assert res_count == 2 # 15 and 25 are > 12
print("Cleartext extremes passed!")

compiler = fhe.Compiler(test_extremes, {"a": "encrypted", "threshold": "clear"})
inputset = [([0, 0, 0, 0], 5), ([15, 5, 25, 10], 12)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(a, threshold)
assert enc_res[0] == res_max
assert enc_res[1] == res_min
assert enc_res[2] == res_rng
assert enc_res[3] == res_count
print("✅ Encrypted extremes passed!")

## Quantiles and Distributions
Functions covered: `array_median`, `array_percentile`, `array_histogram`, `array_mode`

**Note on Sorting in FHE:** `array_median` and `array_percentile` require sorting the array cryptographically. Thus, bounds must be provided to construct the underlying bitonic sorting network.

In [ ]:
from concrete_fhe_toolkit.stats import array_median, array_percentile, array_histogram, array_mode

def test_distributions(a: list[int]):
    median = array_median(a, min_value=0, max_value=30)
    percentile_75 = array_percentile(a, 75, min_value=0, max_value=30)
    hist = array_histogram(a, min_value=0, max_value=30)
    mode = array_mode(a, min_value=0, max_value=30)
    return median, percentile_75, hist, mode

# a must be a power of 2 for the underlying bitonic sort!
a = [10, 20, 20, 30] 

res_med, res_perc, res_hist, res_mode = test_distributions(a)

assert res_med == 20
assert res_perc == 20 # 75th percentile
assert res_hist[20] == 2 # 20 appears twice in the histogram (index 20)
assert res_mode == 20 # Most frequent element
print("Cleartext distributions passed!")

compiler = fhe.Compiler(test_distributions, {"a": "encrypted"})
inputset = [([0, 0, 0, 0],), ([10, 20, 20, 30],)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(a)
assert enc_res[0] == res_med
assert enc_res[1] == res_perc
assert list(enc_res[2])[20] == 2
assert enc_res[3] == res_mode
print("✅ Encrypted distributions passed!")

## Preprocessing
Functions covered: `array_normalize`
Normalizing data inside FHE is highly useful for feeding outputs from one network layer into another layer requiring rigidly bounded inputs.

In [ ]:
from concrete_fhe_toolkit.stats import array_normalize

def test_normalize(a: list[int], mean: int):
    # normalizes elements: (x - mean) // scale
    return array_normalize(a, mean=mean, scale=5)

a = [10, 15, 20]
res_norm = test_normalize(a, mean=15)

# (10-15)//5 = -1
# (15-15)//5 = 0
# (20-15)//5 = 1
assert res_norm == [-1, 0, 1]
print("Cleartext normalize passed!")

compiler = fhe.Compiler(test_normalize, {"a": "encrypted", "mean": "encrypted"})
inputset = [([0, 0, 0], 0), ([10, 15, 20], 15)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(a, 15)
assert list(enc_res) == res_norm
print("✅ Encrypted normalize passed!")